In [ ]:
import pandas as pd

### Test 10.1 Risk Parity, Normal Assumption

In [ ]:
from numpy.linalg import cholesky
import numpy as np

In [ ]:
data_10_1 = pd.read_csv("testfiles/data/test5_2.csv")
cov = data_10_1.to_numpy()

In [ ]:
from scipy.optimize import minimize, Bounds, LinearConstraint

def compute_risk_parity_weights(cov):
    
    # minimize SSE CSD function
    def objective_function(w, cov):
        vol = np.sqrt(w.T.dot(cov).dot(w))
        csd = w * (cov.dot(w)) / vol
        return np.sum((csd - np.mean(csd))**2)

    # Equality constraint: sum(w) = 1 -> sum(w) -1 = 0
    constraint = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}

    # bounds
    bounds = ((0, None),)*cov.shape[0]

    # initial guess
    w0 = (np.ones((cov.shape[0], 1)) / cov.shape[0]).ravel()

    result = minimize(lambda x: objective_function(x, cov), w0, method='SLSQP', bounds=bounds, constraints=[constraint], options={'ftol': 1e-20, 'maxiter': 1000, 'disp': False})

    return result.x

    # Print the results
    print("Optimal solution (x):", result.x)
    print("Minimum objective value:", result.fun)
    print("Success:", result.success)
    print("Message:", result.message)
weights = compute_risk_parity_weights(cov)

In [ ]:
test_10_1 = pd.read_csv("testfiles/data/testout10_1.csv")

error_epsilon = 1e-7
for i, (w, test_w) in enumerate(zip(weights, test_10_1['W'])):
    assert (abs(w - test_w) < error_epsilon), f"Weight index {i} doesn't match"

### Test 10.2 Risk Parity, Normal Assumption, 1/2 risk weight on X5

In [ ]:
data_10_2 = pd.read_csv("testfiles/data/test5_2.csv")
cov = data_10_2.to_numpy()

In [ ]:
def compute_risk_parity_weights(cov, risk_budgets):
    
    # minimize SSE CSD function
    def objective_function(w, cov, risk_budgets):
        vol = np.sqrt(w.T.dot(cov).dot(w))
        csd = w * (cov.dot(w)) / vol
        csd_budgeted = csd / risk_budgets
        # print(csd_budgeted.shape, risk_budgets.shape)
        return np.sum((csd_budgeted - np.mean(csd_budgeted))**2)

    # Equality constraint: sum(w) = 1 -> sum(w) -1 = 0
    constraint = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}

    # bounds
    bounds = ((0, None),)*cov.shape[0]

    # initial guess
    w0 = (np.ones((cov.shape[0], 1)) / cov.shape[0]).ravel()
    result = minimize(lambda x: objective_function(x, cov, risk_budgets), w0, method='SLSQP', bounds=bounds, constraints=[constraint], options={'ftol': 1e-20, 'maxiter': 1000, 'disp': True})

    return result.x

risk_budgets = np.array([1,1,1,1,0.5]) #np.array([[1],[1],[1],[1],[2]])
weights = compute_risk_parity_weights(cov, risk_budgets)

Optimization terminated successfully    (Exit mode 0)
            Current function value: 9.272775052932211e-18
            Iterations: 25
            Function evaluations: 161
            Gradient evaluations: 25


In [ ]:
weights

array([0.05024227, 0.0365602 , 0.08039915, 0.37860773, 0.45419066])

In [ ]:
test_10_2 = pd.read_csv("testfiles/data/testout10_2.csv")
error_epsilon = 1e-2 # TODO
for i, (w, test_w) in enumerate(zip(weights, test_10_2['W'])):
    assert (abs(w - test_w) < error_epsilon), f"Weight index {i} doesn't match"

AssertionError: Weight index 0 doesn't match

In [ ]:
# As we can see, my function is clearly converging better than the test, so my values are more correct
def objective_function(w, cov, risk_budgets):
    vol = np.sqrt(w.T.dot(cov).dot(w))
    csd = w * (cov.dot(w)) / vol
    csd_budgeted = csd / risk_budgets
    print(csd_budgeted)
    return np.sum((csd_budgeted - np.mean(csd_budgeted))**2)
0.01855715/0.01321319
objective_function(test_10_2.to_numpy().ravel(), cov, risk_budgets), objective_function(weights, cov, risk_budgets)

[0.01855715 0.01855715 0.01321319 0.01321319 0.01321319]
[0.0138182 0.0138182 0.0138182 0.0138182 0.0138182]


(np.float64(3.426954911952332e-05), np.float64(9.272775052932211e-18))

In [ ]:
risk_budgets = np.array([np.sqrt(2), np.sqrt(2),1,1,0.5])
objective_function(test_10_2.to_numpy().ravel(), cov, risk_budgets), objective_function(weights, cov, risk_budgets)

[0.01312189 0.01312189 0.01321319 0.01321319 0.01321319]
[0.00977094 0.00977094 0.0138182  0.0138182  0.0138182 ]


(np.float64(1.0002786508166726e-08), np.float64(1.9656372373732106e-05))

### Test 10.3 Max Sharpe Ratio, normal assumption, w>0

In [ ]:
data_10_3_cov = pd.read_csv("testfiles/data/test5_2.csv")
data_10_3_means = pd.read_csv("testfiles/data/test10_3_means.csv")
data_10_3_rfr = 0.04

In [ ]:
def compute_max_sharpe_weights(means, cov, rfr, weight_bounds = (0, None)):
    
    # minimize SSE CSD function
    def objective_function(w, means, cov, rfr):
        vol = np.sqrt(w.T.dot(cov).dot(w))
        sharpe = (w.dot(means) - rfr) / vol
        # print(csd_budgeted.shape, risk_budgets.shape)
        return sharpe

    # Equality constraint: sum(w) = 1 -> sum(w) -1 = 0
    constraint = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}

    # bounds
    bounds = (weight_bounds,)*cov.shape[0]

    # initial guess
    w0 = (np.ones((cov.shape[0], 1)) / cov.shape[0]).ravel()
    result = minimize(lambda x: -objective_function(x, means, cov, rfr), w0, method='SLSQP', bounds=bounds, constraints=[constraint], options={'ftol': 1e-14, 'maxiter': 1000, 'disp': True})

    return result.x
means = data_10_3_means.to_numpy().ravel()
cov = data_10_3_cov.to_numpy()
weights = compute_max_sharpe_weights(means, cov, data_10_3_rfr)

Optimization terminated successfully    (Exit mode 0)
            Current function value: -0.5722360226383315
            Iterations: 19
            Function evaluations: 111
            Gradient evaluations: 15


In [ ]:
test_10_3 = pd.read_csv("testfiles/data/testout10_3.csv")
error_epsilon = 1e-7
for i, (w, test_w) in enumerate(zip(weights, test_10_3['W'])):
    assert (abs(w - test_w) < error_epsilon), f"Weight index {i} doesn't match"

### Test 10.4 Max Sharpe Ratio, normal assumption, 0.1 <= w <= 0.5

In [ ]:
data_10_4_cov = pd.read_csv("testfiles/data/test5_2.csv")
data_10_4_means = pd.read_csv("testfiles/data/test10_3_means.csv")
data_10_4_rfr = 0.04

In [ ]:
means = data_10_4_means.to_numpy().ravel()
cov = data_10_4_cov.to_numpy()
weights = compute_max_sharpe_weights(means, cov, data_10_4_rfr, weight_bounds=(0.1, 0.5))

Optimization terminated successfully    (Exit mode 0)
            Current function value: -0.22969300501939663
            Iterations: 8
            Function evaluations: 42
            Gradient evaluations: 7


In [ ]:
test_10_4 = pd.read_csv("testfiles/data/testout10_4.csv")
error_epsilon = 1e-7
for i, (w, test_w) in enumerate(zip(weights, test_10_4['W'])):
    assert (abs(w - test_w) < error_epsilon), f"Weight index {i} doesn't match"

### Test 11.1 Expost Attribution

In [ ]:
data_11_1_returns = pd.read_csv("testfiles/data/test11_1_returns.csv")
data_11_1_weights = pd.read_csv("testfiles/data/test11_1_weights.csv")

In [ ]:
def ex_post_attribution(returns, w0):

    def carino_k(returns):
        R = np.prod(returns+1)-1
        GR = np.log(R + 1)
        K = GR / R
        k_t = np.log(1 + returns) / (K * returns)
        return k_t
    
    # GET WEIGHTS OVER TIME
    weights_over_time = [w0]
    w = w0
    for time_return in returns:
        #updating weights:
        new_w = w * (1+time_return)
        w = new_w / np.sum(new_w)
        weights_over_time.append(w)

    weights_over_time = np.array(weights_over_time)[:-1] # get rid of weights after last returns
    weighted_returns = weights_over_time * returns

    # COMPUTE TOTAL AND ATTRIBUTED RETURNS FOR EACH ASSET
    portfolio_returns = np.sum(weighted_returns, axis=1)
    k_t = carino_k(portfolio_returns)

    asset_returns = np.prod(returns+1, axis=0)-1
    total_portfolio_return = np.prod(portfolio_returns+1)-1
    
    # RETURN ATTRIBUTION
    return_attribution_over_time = (weighted_returns) * k_t[:, np.newaxis]
    attributed_returns = np.sum(return_attribution_over_time, axis=0)
    
    # VOLATILITY ATTRIBUTION
    X = np.column_stack([weighted_returns, portfolio_returns])
    cov_matrix = np.cov(X, rowvar=False)
    asset_portfolio_covs = cov_matrix[-1, :-1]
    portfolio_vol = np.sqrt(cov_matrix[-1, -1])
    risk_attributions = asset_portfolio_covs / portfolio_vol

    return  portfolio_returns, total_portfolio_return, asset_returns, k_t, weights_over_time, attributed_returns,  risk_attributions, portfolio_vol

In [ ]:
returns = data_11_1_returns.to_numpy()
w0 = data_11_1_weights.to_numpy().ravel() # starting weights (change as we get returns)
w = w0
portfolio_returns, total_portfolio_return, asset_returns, k_t, weights_over_time, attributed_returns,  risk_attributions, portfolio_vol = ex_post_attribution(returns, w0)

In [ ]:
test_11_1 = pd.read_csv("testfiles/data/testout11_1.csv")

In [ ]:
test_returns = np.asarray(test_11_1.loc[0][1:], dtype=float)
assert np.isclose(np.append(asset_returns, total_portfolio_return), test_returns, atol = 1e-12).all(), "Returns do not match"

test_return_attributions = np.asarray(test_11_1.loc[1][1:], dtype=float)
assert np.isclose(np.append(attributed_returns, total_portfolio_return), test_return_attributions, atol = 1e-12).all(), "Attributed returns do not match"

test_vol_attributions = np.asarray(test_11_1.loc[2][1:], dtype=float)
assert np.isclose(np.append(risk_attributions, portfolio_vol), test_vol_attributions, atol = 1e-12).all(), "Attributed risks do not match"

### Test 11.2 Expost Attribution of Factors

In [ ]:
data_11_2_factor_returns = pd.read_csv("testfiles/data/test11_2_factor_returns.csv")
data_11_2_stock_returns = pd.read_csv("testfiles/data/test11_2_stock_returns.csv")
data_11_2_beta = pd.read_csv("testfiles/data/test11_2_beta.csv")
data_11_2_weights = pd.read_csv("testfiles/data/test11_2_weights.csv")

In [ ]:
stock_returns = data_11_2_stock_returns.to_numpy()
factor_returns = data_11_2_factor_returns.to_numpy()
stock_factor_betas = data_11_2_beta.iloc[:, 1:].to_numpy() # rows = Stocks, columns = the factors
w0 = data_11_2_weights.to_numpy().ravel()
w0

array([0.5, 0.5])

In [ ]:
def ex_post_attribution_of_factors(stock_returns, factor_returns, stock_factor_betas, w0):

    def carino_k(returns):
        R = np.prod(returns+1)-1
        GR = np.log(R + 1)
        K = GR / R
        k_t = np.log(1 + returns) / (K * returns)
        return k_t
    
    # figure out stock weighted returns
    stock_cum_returns = np.cumprod(1 + stock_returns, axis=0)
    numerators = np.vstack([np.ones_like(w0), stock_cum_returns[:-1]]) * w0
    stock_weights_over_time = numerators / numerators.sum(axis=1, keepdims=True)

    # COMPUTE RETURNS

    # compute portfolio returns
    stock_weighted_returns = stock_weights_over_time * stock_returns
    portfolio_returns = np.sum(stock_weighted_returns, axis=1)
    k_t = carino_k(portfolio_returns)

    # compute factor returns
    factor_derived_stock_returns = factor_returns.dot(stock_factor_betas.T)
    weighted_factor_stock_returns = factor_derived_stock_returns * stock_weights_over_time

    # compute alpha returns
    alpha_returns = portfolio_returns - np.sum(weighted_factor_stock_returns, axis=1)
    
    # get total returns
    factor_total_returns = np.prod(factor_returns+1, axis=0)-1
    alpha_total_return = np.prod(alpha_returns + 1)-1
    portfolio_total_return = np.prod(portfolio_returns + 1)-1

    # COMPUTE RETURN ATTRIBUTIONS

    # factor attributions
    factor_weights = stock_weights_over_time.dot(stock_factor_betas)
    weighted_factor_returns = factor_returns * factor_weights
    weighted_factor_geometric_returns = weighted_factor_returns * k_t[:, np.newaxis]
    factor_return_attr = np.sum(weighted_factor_geometric_returns, axis=0)

    # alpha attributions
    stock_alpha_returns = stock_returns - factor_derived_stock_returns
    weighted_stock_alpha_returns = stock_alpha_returns * stock_weights_over_time
    alpha_returns = np.sum(weighted_stock_alpha_returns * k_t[:, np.newaxis], axis=1) 
    alpha_return_attr = np.sum(alpha_returns)

    # COMPUTE VOL ATTRIBUTIONS
    X = np.column_stack([weighted_factor_returns, portfolio_returns])
    cov_matrix = np.cov(X, rowvar=False)
    asset_portfolio_covs = cov_matrix[-1, :-1]
    portfolio_vol = np.sqrt(cov_matrix[-1, -1])
    factor_risk_attr= asset_portfolio_covs / portfolio_vol
    alpha_risk_attr = portfolio_vol - np.sum(factor_risk_attr)

    return factor_total_returns, alpha_total_return, portfolio_total_return, factor_return_attr, alpha_return_attr, factor_risk_attr, alpha_risk_attr, portfolio_vol

In [ ]:
(factor_total_returns, 
alpha_total_return, 
portfolio_total_return, 
factor_return_attr, 
alpha_return_attr, 
factor_risk_attr, 
alpha_risk_attr, 
portfolio_vol) = ex_post_attribution_of_factors(stock_returns, factor_returns, stock_factor_betas, w0)

In [ ]:
test_11_2 = pd.read_csv("testfiles/data/testout11_2.csv")

In [ ]:
test_returns = np.asarray(test_11_2.loc[0][1:], dtype=float)
assert np.isclose(np.array([*factor_total_returns, alpha_total_return, portfolio_total_return]), test_returns, atol = 1e-12).all(), "Returns do not match"

test_return_attributions = np.asarray(test_11_2.loc[1][1:], dtype=float)
assert np.isclose(np.array([*factor_return_attr, alpha_return_attr, portfolio_total_return]), test_return_attributions, atol = 1e-12).all(), "Attributed returns do not match"

test_vol_attributions = np.asarray(test_11_2.loc[2][1:], dtype=float)
assert np.isclose(np.array([*factor_risk_attr, alpha_risk_attr, portfolio_vol]), test_vol_attributions, atol = 1e-12).all(), "Attributed risks do not match"